In [ ]:
import math
import os
import random
from typing import Callable, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def set_seed(seed: int = 1):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
class PSAFM(nn.Module):

    def __init__(self, kernel_size: int = 3, lambda_: float = 1e-4, gamma: float = 0.5):
        super().__init__()
        padding = kernel_size // 2
        self.mp = nn.MaxPool2d(kernel_size=kernel_size, stride=1, padding=padding)
        self.ap = nn.AvgPool2d(kernel_size=kernel_size, stride=1, padding=padding)
        self.lambda_ = lambda_
        self.gamma = gamma

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        fm1 = self.mp(x)
        fa1 = self.ap(x)
        fm2 = self.mp(x + fm1)
        fa2 = self.ap(x + fa1)
        fad = fm2 + fa2
        #Channel-statistical 3-D attention
        b, c, h, w = fad.shape
        n = h * w - 1
        mu = fad.mean(dim=(2, 3), keepdim=True)
        x_square = (fad - mu).pow(2)
        s = x_square.sum(dim=(2, 3), keepdim=True)
        y = x_square / (4.0 * (s / n + self.lambda_)) + self.gamma
        return fad * torch.tanh(y)
_demo = PSAFM()
_x = torch.randn(2, 16, 32, 32)
_y = _demo(_x)
assert _y.shape == _x.shape
assert sum(p.numel() for p in _demo.parameters()) == 0
print(f"PSAFM OK | in {_x.shape} -> out {_y.shape} | params = 0")

PSAFM OK | in torch.Size([2, 16, 32, 32]) -> out torch.Size([2, 16, 32, 32]) | params = 0


In [ ]:
class SE(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)
        return x * w
class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )

    def forward(self, x):
        avg = self.mlp(F.adaptive_avg_pool2d(x, 1))
        mx = self.mlp(F.adaptive_max_pool2d(x, 1))
        return x * torch.sigmoid(avg + mx)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, spatial_kernel: int = 7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)

    def forward(self, x):
        return self.sa(self.ca(x))

class SimAM(nn.Module):

    def __init__(self, lambda_: float = 1e-4):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        b, c, h, w = x.shape
        n = h * w - 1
        mu = x.mean(dim=(2, 3), keepdim=True)
        d = (x - mu).pow(2)
        s = d.sum(dim=(2, 3), keepdim=True)
        e_inv = d / (4.0 * (s / n + self.lambda_)) + 0.5
        return x * torch.sigmoid(e_inv)


def build_attention(name: str, channels: int) -> Optional[nn.Module]:
    name = name.lower()
    if name in ("none", "original", ""):
        return None
    if name == "se":
        return SE(channels)
    if name == "cbam":
        return CBAM(channels)
    if name == "simam":
        return SimAM()
    if name == "psafm":
        return PSAFM()
    raise ValueError(f"Unknown attention module: {name}")

In [ ]:
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, 3, stride=stride, padding=1, bias=False)


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, 1, stride=stride, bias=False)

#Standard ResNet
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, attention: str = "none"):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.attn = build_attention(attention, planes * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                conv1x1(in_planes, planes * self.expansion, stride),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.attn is not None:
            out = self.attn(out)
        out = out + self.shortcut(x)
        return F.relu(out)


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, attention: str = "none"):
        super().__init__()
        self.conv1 = conv1x1(in_planes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.attn = build_attention(attention, planes * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                conv1x1(in_planes, planes * self.expansion, stride),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.attn is not None:
            out = self.attn(out)
        out = out + self.shortcut(x)
        return F.relu(out)


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10, attention: str = "none"):
        super().__init__()
        self.in_planes = 16
        self.attention = attention
        self.conv1 = conv3x3(3, 16)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64 * block.expansion, num_classes)
        self._init_weights()

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s, self.attention))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out).flatten(1)
        return self.fc(out)


def ResNet20(num_classes=10, attention="none"):
    return ResNet(BasicBlock, [3, 3, 3], num_classes, attention)

def ResNet56(num_classes=10, attention="none"):
    return ResNet(BasicBlock, [9, 9, 9], num_classes, attention)

def ResNet110(num_classes=10, attention="none"):
    return ResNet(BasicBlock, [18, 18, 18], num_classes, attention)

def ResNet164(num_classes=10, attention="none"):
    return ResNet(Bottleneck, [18, 18, 18], num_classes, attention)

In [ ]:
#Pre-ResNet (identity mappings)
class PreBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, attention: str = "none"):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes)
        self.attn = build_attention(attention, planes * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = conv1x1(in_planes, planes * self.expansion, stride)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        if self.attn is not None:
            out = self.attn(out)
        return out + self.shortcut(x)


class PreBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, attention: str = "none"):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = conv1x1(in_planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn3 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.attn = build_attention(attention, planes * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = conv1x1(in_planes, planes * self.expansion, stride)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.conv3(F.relu(self.bn3(out)))
        if self.attn is not None:
            out = self.attn(out)
        return out + self.shortcut(x)


class PreResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10, attention: str = "none"):
        super().__init__()
        self.in_planes = 16
        self.attention = attention
        self.conv1 = conv3x3(3, 16)
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.bn = nn.BatchNorm2d(64 * block.expansion)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64 * block.expansion, num_classes)
        self._init_weights()

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s, self.attention))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn(out))
        out = self.avgpool(out).flatten(1)
        return self.fc(out)


def PreResNet20(num_classes=10, attention="none"):
    return PreResNet(PreBasicBlock, [3, 3, 3], num_classes, attention)


def PreResNet56(num_classes=10, attention="none"):
    return PreResNet(PreBasicBlock, [9, 9, 9], num_classes, attention)


def PreResNet110(num_classes=10, attention="none"):
    return PreResNet(PreBasicBlock, [18, 18, 18], num_classes, attention)


def PreResNet164(num_classes=10, attention="none"):
    return PreResNet(PreBottleneck, [18, 18, 18], num_classes, attention)

In [ ]:
#MobileNetV2
class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio, attention: str = "none"):
        super().__init__()
        assert stride in (1, 2)
        hidden = int(round(inp * expand_ratio))
        self.use_res = stride == 1 and inp == oup
        layers = []
        if expand_ratio != 1:
            layers.extend([
                conv1x1(inp, hidden),
                nn.BatchNorm2d(hidden),
                nn.ReLU6(inplace=True),
            ])
        layers.extend([
            nn.Conv2d(hidden, hidden, 3, stride, 1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),
            conv1x1(hidden, oup),
            nn.BatchNorm2d(oup),
        ])
        self.conv = nn.Sequential(*layers)
        self.attn = build_attention(attention, oup)

    def forward(self, x):
        out = self.conv(x)
        if self.attn is not None:
            out = self.attn(out)
        if self.use_res:
            return x + out
        return out


class MobileNetV2(nn.Module):
    def __init__(self, num_classes=10, width_mult=1.0, attention: str = "none"):
        super().__init__()
        # CIFAR-adapted setting (smaller first stride)
        cfg = [
            # t, c, n, s
            [1, 16, 1, 1],
            [6, 24, 2, 1],
            [6, 32, 3, 2],
            [6, 64, 4, 2],
            [6, 96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]
        input_channel = int(32 * width_mult)
        last_channel = int(1280 * width_mult) if width_mult > 1.0 else 1280

        features = [
            nn.Sequential(
                conv3x3(3, input_channel, stride=1),
                nn.BatchNorm2d(input_channel),
                nn.ReLU6(inplace=True),
            )
        ]
        for t, c, n, s in cfg:
            output_channel = int(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(
                    InvertedResidual(input_channel, output_channel, stride, t, attention)
                )
                input_channel = output_channel

        features.append(
            nn.Sequential(
                conv1x1(input_channel, last_channel),
                nn.BatchNorm2d(last_channel),
                nn.ReLU6(inplace=True),
            )
        )
        self.features = nn.Sequential(*features)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(last_channel, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        return self.classifier(x)


MODEL_ZOO: dict[str, Callable] = {
    "resnet20": ResNet20,
    "resnet56": ResNet56,
    "resnet110": ResNet110,
    "resnet164": ResNet164,
    "preresnet20": PreResNet20,
    "preresnet56": PreResNet56,
    "preresnet110": PreResNet110,
    "preresnet164": PreResNet164,
    "mobilenetv2": lambda num_classes=10, attention="none": MobileNetV2(
        num_classes=num_classes, attention=attention
    ),
}


def create_model(name: str, num_classes: int = 10, attention: str = "none") -> nn.Module:
    key = name.lower().replace("-", "").replace("_", "")
    if key not in MODEL_ZOO:
        raise ValueError(f"Unknown model '{name}'. Choose from: {list(MODEL_ZOO)}")
    return MODEL_ZOO[key](num_classes=num_classes, attention=attention)

 Data loaders (CIFAR-10 / CIFAR-100)

In [ ]:
def get_dataloaders(
    dataset: str = "cifar10",
    data_root: str = "./data",
    batch_size: int = 128,
    num_workers: int = 2,
):
    dataset = dataset.lower()
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    if dataset == "cifar100":
        mean = (0.5071, 0.4867, 0.4408)
        std = (0.2675, 0.2565, 0.2761)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    if dataset == "cifar10":
        train_set = datasets.CIFAR10(data_root, train=True, download=True, transform=train_tf)
        test_set = datasets.CIFAR10(data_root, train=False, download=True, transform=test_tf)
        num_classes = 10
    elif dataset == "cifar100":
        train_set = datasets.CIFAR100(data_root, train=True, download=True, transform=train_tf)
        test_set = datasets.CIFAR100(data_root, train=False, download=True, transform=test_tf)
        num_classes = 100
    else:
        raise ValueError("dataset should be 'cifar10' or 'cifar100'")

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True,
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return train_loader, test_loader, num_classes

Train / evaluate helpers

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device=DEVICE) -> float:
    model.eval()
    correct = total = 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        pred = logits.argmax(dim=1)
        correct += (pred == targets).sum().item()
        total += targets.size(0)
    return 100.0 * correct / total


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_flops(model: nn.Module, input_size=(1, 3, 32, 32), device=DEVICE) -> float:
    """Approximate flops via fvcore if available else return nan."""
    try:
        from fvcore.nn import FlopCountAnalysis
        model = model.to(device).eval()
        x = torch.randn(*input_size, device=device)
        flops = FlopCountAnalysis(model, x).total()
        return flops / 1e9
    except Exception:
        return float("nan")


def train_one_run(
    model_name: str = "preresnet56",
    attention: str = "psafm",
    dataset: str = "cifar10",
    epochs: int = 100,
    batch_size: int = 128,
    lr: float = 0.1,
    weight_decay: float = 5e-4,
    seed: int = 1,
    data_root: str = "./data",
    lr_milestones: Optional[list] = None,
    print_every: int = 10,
):
    set_seed(seed)
    train_loader, test_loader, num_classes = get_dataloaders(
        dataset, data_root, batch_size
    )
    model = create_model(model_name, num_classes, attention).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay
    )
    if lr_milestones is None:
        lr_milestones = [int(epochs * 0.5), int(epochs * 0.75)]
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=lr_milestones, gamma=0.1
    )

    params_m = count_parameters(model) / 1e6
    print(
        f"{model_name} + {attention} | {dataset} | "
        f"params={params_m:.6f}M | device={DEVICE}"
    )

    best_acc = 0.0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct = total = 0
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * targets.size(0)
            correct += (logits.argmax(1) == targets).sum().item()
            total += targets.size(0)

        scheduler.step()
        train_acc = 100.0 * correct / total
        test_acc = evaluate(model, test_loader)
        best_acc = max(best_acc, test_acc)
        history.append({"epoch": epoch, "train_acc": train_acc, "test_acc": test_acc})

        if epoch % print_every == 0 or epoch == 1 or epoch == epochs:
            print(
                f"Epoch {epoch:3d}/{epochs} | "
                f"loss={running_loss / total:.4f} | "
                f"train={train_acc:.2f}% | test={test_acc:.2f}% | best={best_acc:.2f}%"
            )

    return {
        "best_acc": best_acc,
        "params_m": params_m,
        "history": history,
        "model": model,
    }

In [ ]:
set_seed(1)
for attn in ["none", "se", "cbam", "simam", "psafm"]:
    m = create_model("preresnet20", num_classes=10, attention=attn)
    x = torch.randn(2, 3, 32, 32)
    y = m(x)
    print(
        f"{attn:8s} | out={tuple(y.shape)} | "
        f"params={count_parameters(m)/1e6:.6f}M"
    )

none     | out=(2, 10) | params=0.272282M
se       | out=(2, 10) | params=0.274298M
cbam     | out=(2, 10) | params=0.275180M
simam    | out=(2, 10) | params=0.272282M
psafm    | out=(2, 10) | params=0.272282M


In [ ]:
MODEL_NAME = "preresnet56"
ATTENTION = "psafm"
DATASET = "cifar10"
EPOCHS = 100
BATCH_SIZE = 128
SEED = 1

result = train_one_run(
    model_name=MODEL_NAME,
    attention=ATTENTION,
    dataset=DATASET,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
)
print(f"\nBest accuracy: {result['best_acc']:.2f}%")
print(f"Parameters:    {result['params_m']:.6f}M")

100%|██████████| 170M/170M [38:38<00:00, 73.5kB/s]


preresnet56 + psafm | cifar10 | params=0.855578M | device=cuda
Epoch   1/100 | loss=1.5616 | train=42.06% | test=53.18% | best=53.18%
Epoch  10/100 | loss=0.5511 | train=81.18% | test=72.23% | best=78.15%
Epoch  20/100 | loss=0.4736 | train=83.97% | test=81.16% | best=81.16%
Epoch  30/100 | loss=0.4514 | train=84.53% | test=79.87% | best=81.16%
Epoch  40/100 | loss=0.4306 | train=85.25% | test=75.65% | best=82.96%
Epoch  50/100 | loss=0.4166 | train=85.72% | test=80.34% | best=83.39%
Epoch  60/100 | loss=0.1382 | train=95.28% | test=90.80% | best=91.18%
Epoch  70/100 | loss=0.1171 | train=95.89% | test=90.54% | best=91.18%
Epoch  80/100 | loss=0.0464 | train=98.54% | test=92.37% | best=92.37%
Epoch  90/100 | loss=0.0302 | train=99.18% | test=92.20% | best=92.44%
Epoch 100/100 | loss=0.0234 | train=99.35% | test=92.24% | best=92.44%

Best accuracy: 92.44%
Parameters:    0.855578M
